# 🎬 Filmverkstan – AI-video i molnet (gratis)

Gör en **rörlig scen av en bild** med AI-modellen **LTX-Video 0.9.5**, på Googles grafikkort.

## Så här gör du
1. **Slå på grafikkortet:** menyn **Körning → Ändra körningstyp → T4 GPU → Spara**.
2. **Kör steg 1–3** genom att klicka på ▶ till vänster om varje ruta, i ordning.
3. I **steg 4** skriver du vad som ska hända och trycker ▶. Första klippet tar ca 30 min (modellerna laddas ner), sedan ca **5 minuter** per klipp.
4. Vill du göra fler klipp? Ändra i steg 3 eller 4 och kör om bara de rutorna.
5. **Steg 5** laddar ner alla klipp till din dator.

> 💡 Gratis-Colab har en tidsgräns per dag. Stäng fliken när du är klar så sparar du tid till nästa gång.

In [ ]:
#@title 1. Installera (tar 1–2 minuter) { display-mode: "form" }
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ Inget grafikkort! Gå till Körning → Ändra körningstyp → välj T4 GPU → Spara. Kör sedan den här rutan igen.")
print("✅ Grafikkort:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "diffusers>=0.35", "transformers", "accelerate", "sentencepiece", "imageio", "imageio-ffmpeg"], check=True)
print("✅ Klart! Fortsätt med steg 2.")

In [ ]:
#@title 2. Förbered verktyget { display-mode: "form" }
import gc, os, time, base64, torch
from PIL import Image
from IPython.display import HTML, display

REPO = "Lightricks/LTX-Video"          # textmodellen (T5)
VREPO = "Lightricks/LTX-Video-0.9.5"   # videomodellen – nyare version med bättre rörelser
DTYPE = torch.bfloat16        # textmodellen (T5 klarar inte float16)
VDTYPE = torch.float16        # videomodellen – T4-kortet har sparsam beräkning bara för float16
OUT = "/content/klipp"
os.makedirs(OUT, exist_ok=True)
NEGATIVE = ("worst quality, blurry, distorted face, deformed hands, extra fingers, inconsistent motion, "
            "jittery, flickering, static, text, watermark")
SIZES = {"Stående 9:16 (TikTok/Reels)": (480, 832), "Liggande 16:9 (YouTube)": (832, 480), "Fyrkantig 1:1": (640, 640)}
_video_pipe = None

def _free():
    gc.collect(); torch.cuda.empty_cache()

def _fit(img, w, h):
    img = img.convert("RGB")
    s = max(w / img.width, h / img.height)
    img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
    l, t = (img.width - w) // 2, (img.height - h) // 2
    return img.crop((l, t, l + w, t + h))

def _encode(prompt):
    """Steg A: läs in texten med textmodellen, och släpp den sedan (så minnet räcker).
    Videomodellen flyttas undan till RAM under tiden, så de turas om på grafikkortet."""
    if _video_pipe is not None:
        _video_pipe.to("cpu"); _free()
    from transformers import T5EncoderModel, AutoTokenizer
    from diffusers import LTXImageToVideoPipeline, FlowMatchEulerDiscreteScheduler
    # Samma ordlista som LTX använder, men färdigkonverterad (undviker en bugg i nya transformers)
    tok = AutoTokenizer.from_pretrained("google-t5/t5-base")
    te = T5EncoderModel.from_pretrained(REPO, subfolder="text_encoder", torch_dtype=DTYPE,
                                        device_map="cuda", low_cpu_mem_usage=True)  # direkt till grafikkortet
    p = LTXImageToVideoPipeline(scheduler=FlowMatchEulerDiscreteScheduler(), vae=None,
                                text_encoder=te, tokenizer=tok, transformer=None)
    with torch.no_grad():
        emb = p.encode_prompt(prompt, NEGATIVE, do_classifier_free_guidance=True, device="cuda")
    del p, te; _free()
    emb = tuple(x.to(VDTYPE) if x.is_floating_point() else x for x in emb)
    if _video_pipe is not None:
        _video_pipe.to("cuda")
    return emb

def _video():
    """Steg B: videomodellen (laddas en gång och sparas i minnet)."""
    global _video_pipe
    if _video_pipe is None:
        from diffusers import LTXConditionPipeline
        _video_pipe = LTXConditionPipeline.from_pretrained(
            VREPO, text_encoder=None, tokenizer=None, torch_dtype=VDTYPE).to("cuda")
        _video_pipe.vae.enable_tiling()
    return _video_pipe

def make_clip(image_path, prompt, format_name, seconds, steps, seed):
    from diffusers.utils import export_to_video
    w, h = SIZES[format_name]
    frames = 8 * round(seconds * 24 / 8) + 1          # LTX vill ha 8·n+1 bildrutor
    t0 = time.time()
    print("📝 Läser in din beskrivning…"); pe, pm, ne, nm = _encode(prompt)
    print("🎞️ Laddar videomodellen…");   pipe = _video()
    gen = torch.Generator("cuda").manual_seed(int(seed)) if seed >= 0 else None
    def cb(p, i, t, k):
        if i % 5 == 0 or i == steps - 1:
            print(f"   steg {i+1}/{steps}  ({time.time()-t0:.0f} s)")
        return k
    print("🎬 Skapar videon…")
    with torch.no_grad():
        out = pipe(image=_fit(Image.open(image_path), w, h),
                   prompt_embeds=pe, prompt_attention_mask=pm,
                   negative_prompt_embeds=ne, negative_prompt_attention_mask=nm,
                   width=w, height=h, num_frames=frames, num_inference_steps=int(steps),
                   guidance_scale=3.0, generator=gen, callback_on_step_end=cb).frames[0]
    path = f"{OUT}/klipp_{time.strftime('%H%M%S')}.mp4"
    export_to_video(out, path, fps=24)
    del pe, pm, ne, nm; _free()
    print(f"✅ Klart på {(time.time()-t0)/60:.1f} min → {path}")
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video controls loop autoplay style="max-height:520px" src="data:video/mp4;base64,{b64}"></video>'))
    return path

print("✅ Verktyget är redo. Fortsätt med steg 3.")

In [ ]:
#@title 3. Ladda upp startbild { display-mode: "form" }
#@markdown Tryck ▶ och välj en bild från datorn – **eller** klistra in en bildlänk här (t.ex. från Runway):
Bildlank = "" #@param {type:"string"}
import urllib.request
if Bildlank.strip():
    BILD = "/content/startbild.png"
    urllib.request.urlretrieve(Bildlank.strip(), BILD)
else:
    from google.colab import files
    up = files.upload()
    BILD = "/content/" + list(up)[0]
    with open(BILD, "wb") as f:
        f.write(up[list(up)[0]])
from IPython.display import display
from PIL import Image
im = Image.open(BILD); im.thumbnail((360, 360)); display(im)
print("✅ Bild:", BILD)

In [ ]:
#@title 4. Skapa video { display-mode: "form" }
#@markdown **Vad ska hända?** Skriv på engelska och beskriv *rörelsen*. Tips: 3 sekunder ger skarpast resultat.
Beskrivning = "The old man raises the axe and splits the log in one strong motion, wood chips fly, camera slowly pushes in" #@param {type:"string"}
Format = "Stående 9:16 (TikTok/Reels)" #@param ["Stående 9:16 (TikTok/Reels)", "Liggande 16:9 (YouTube)", "Fyrkantig 1:1"]
Sekunder = 3 #@param {type:"slider", min:2, max:6, step:1}
#@markdown Fler steg = bättre kvalitet men långsammare (25 = snabbt, 40 = bra).
Kvalitet = 40 #@param {type:"slider", min:20, max:50, step:5}
#@markdown Seed: -1 = slumpa. Samma tal ger samma resultat igen.
Seed = -1 #@param {type:"integer"}
klipp = make_clip(BILD, Beskrivning, Format, Sekunder, Kvalitet, Seed)

In [ ]:
#@title 5. Ladda ner alla klipp { display-mode: "form" }
import shutil
from google.colab import files
zipfil = shutil.make_archive("/content/filmverkstan_klipp", "zip", OUT)
files.download(zipfil)

## Färdiga beskrivningar för skogsmannen
Kopiera in i steg 4, med rätt bild uppladdad i steg 3:

| Scen | Beskrivning |
|---|---|
| Morgon vid stugan | `The old man opens the cabin door and takes a slow deep breath, eyes closed, morning mist drifts between the pine trees, birds flutter on a branch` |
| Trädgården | `The old man gently picks blueberries and drops them into a wooden bowl, leaves sway softly, warm sunlight, steam rises from a clay mug` |
| Vedhuggning | `The old man raises the axe and splits the log in one strong motion, wood chips fly, camera slowly pushes in` |
| Sjön | `The old man sits calmly by the misty lake and lifts the steaming mug to drink, mist slowly drifts over the water` |
| Leendet | `The old man looks into the camera and slowly smiles warmly, sunbeams flicker through the trees behind him, gentle breeze in his beard` |

## Om något går fel
- **"CUDA out of memory"** → välj färre sekunder eller lägre kvalitet. Du kan också välja **Körning → Starta om sessionen** och köra steg 1–4 igen.
- **"Inget grafikkort"** → Körning → Ändra körningstyp → T4 GPU.
- **Gränsen för gratis-tid är nådd** → vänta till nästa dag.